# Road Accidents Prediction - XGBoost Optimized
## Pure XGBoost Pipeline with Hyperparameter Tuning

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

## 1. DATA LOADING

In [2]:
# Load datasets
TRAIN_PATH = r'C:\Users\ASUS\Desktop\Sem04\Coding\sem04_Codes\UOM_sem_04\Inputs\Accident\train.csv'
TEST_PATH = r'C:\Users\ASUS\Desktop\Sem04\Coding\sem04_Codes\UOM_sem_04\Inputs\Accident\test.csv'
TARGET_COL = 'accident_risk'
ID_COL = 'id'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nTrain columns: {train_df.columns.tolist()}")

Train shape: (517754, 14)
Test shape: (172585, 13)

Train columns: ['id', 'road_type', 'num_lanes', 'curvature', 'speed_limit', 'lighting', 'weather', 'road_signs_present', 'public_road', 'time_of_day', 'holiday', 'school_season', 'num_reported_accidents', 'accident_risk']


## 2. DATA PREPROCESSING

In [3]:
# Separate features and target
X = train_df.drop(columns=[TARGET_COL, ID_COL])
y = train_df[TARGET_COL]

# Save test IDs for submission
test_ids = test_df[ID_COL].copy()
X_test = test_df.drop(columns=[ID_COL])

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Test features shape: {X_test.shape}")

Features shape: (517754, 12)
Target shape: (517754,)
Test features shape: (172585, 12)


In [4]:
# Drop unnecessary columns
drop_cols = ["road_signs_present", "school_season", "num_lanes", "time_of_day"]

X = X.drop(columns=drop_cols, errors='ignore')
X_test = X_test.drop(columns=drop_cols, errors='ignore')

print(f"Features after dropping: {X.columns.tolist()}")

Features after dropping: ['road_type', 'curvature', 'speed_limit', 'lighting', 'weather', 'public_road', 'holiday', 'num_reported_accidents']


In [5]:
# Identify numeric and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()

print(f"Numeric columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")

Numeric columns: ['curvature', 'speed_limit', 'num_reported_accidents']
Categorical columns: ['road_type', 'lighting', 'weather', 'public_road', 'holiday']


In [7]:
# Impute missing values (XGBoost handles missing values natively, but we'll impute for safety)
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

if num_cols:
    X[num_cols] = num_imputer.fit_transform(X[num_cols])
    X_test[num_cols] = num_imputer.transform(X_test[num_cols])

if cat_cols:
    X[cat_cols] = X[cat_cols].astype("object")
    X_test[cat_cols] = X_test[cat_cols].astype("object")
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
    X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

print("Missing values imputed")
print(f"Missing in X: {X.isnull().sum().sum()}")
print(f"Missing in X_test: {X_test.isnull().sum().sum()}")

Missing values imputed
Missing in X: 0
Missing in X_test: 0


In [8]:
# One-Hot Encoding for categorical features
# XGBoost works well with one-hot encoded features
X = pd.get_dummies(X, columns=cat_cols, drop_first=False)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=False)

# Align test set with train set features
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

print(f"Features after encoding: {X.shape}")
print(f"Test features after encoding: {X_test.shape}")
print(f"Feature names: {X.columns.tolist()}")

Features after encoding: (517754, 16)
Test features after encoding: (172585, 16)
Feature names: ['curvature', 'speed_limit', 'num_reported_accidents', 'road_type_highway', 'road_type_rural', 'road_type_urban', 'lighting_daylight', 'lighting_dim', 'lighting_night', 'weather_clear', 'weather_foggy', 'weather_rainy', 'public_road_False', 'public_road_True', 'holiday_False', 'holiday_True']


In [9]:
# Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

Training set: (414203, 16)
Validation set: (103551, 16)


## 3. XGBOOST BASELINE MODEL

In [10]:
# Train baseline XGBoost model
baseline_model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='rmse',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

print("Training baseline XGBoost model...")
baseline_model.fit(X_train, y_train)

# Evaluate baseline
y_pred_baseline = baseline_model.predict(X_val)
baseline_rmse = np.sqrt(mean_squared_error(y_val, y_pred_baseline))
baseline_r2 = r2_score(y_val, y_pred_baseline)
baseline_mae = mean_absolute_error(y_val, y_pred_baseline)

print(f"\nBaseline XGBoost Performance:")
print(f"  RMSE: {baseline_rmse:.6f}")
print(f"  R²:   {baseline_r2:.6f}")
print(f"  MAE:  {baseline_mae:.6f}")

Training baseline XGBoost model...

Baseline XGBoost Performance:
  RMSE: 0.056467
  R²:   0.884524
  MAE:  0.043875


## 4. HYPERPARAMETER TUNING WITH RANDOMIZEDSEARCHCV

In [11]:
# Define hyperparameter search space
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.5, 1.0]
}

print("Starting RandomizedSearchCV for hyperparameter tuning...")
print("This may take a few minutes...\n")

xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(eval_metric='rmse', random_state=42, n_jobs=-1, verbosity=0),
    param_distributions=param_grid,
    n_iter=40,  # Try 40 combinations
    scoring='r2',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)

print(f"\nBest parameters found:")
for param, value in xgb_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest CV R² Score: {xgb_search.best_score_:.6f}")

Starting RandomizedSearchCV for hyperparameter tuning...
This may take a few minutes...

Fitting 5 folds for each of 40 candidates, totalling 200 fits

Best parameters found:
  subsample: 0.6
  n_estimators: 100
  min_child_weight: 3
  max_depth: 7
  learning_rate: 0.1
  gamma: 0
  colsample_bytree: 0.6

Best CV R² Score: 0.886322


## 5. TRAIN OPTIMIZED MODEL

In [12]:
# Create optimized model with best parameters
optimized_model = XGBRegressor(
    **xgb_search.best_params_,
    eval_metric='rmse',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

print("Training optimized XGBoost model...")
optimized_model.fit(X_train, y_train, verbose=False)

# Evaluate on validation set
y_pred_optimized = optimized_model.predict(X_val)
opt_rmse = np.sqrt(mean_squared_error(y_val, y_pred_optimized))
opt_r2 = r2_score(y_val, y_pred_optimized)
opt_mae = mean_absolute_error(y_val, y_pred_optimized)

print(f"\nOptimized XGBoost Performance (Validation Set):")
print(f"  RMSE: {opt_rmse:.6f}")
print(f"  R²:   {opt_r2:.6f}")
print(f"  MAE:  {opt_mae:.6f}")

# Compare with baseline
rmse_improvement = ((baseline_rmse - opt_rmse) / baseline_rmse) * 100
r2_improvement = ((opt_r2 - baseline_r2) / baseline_r2) * 100

print(f"\nImprovement over Baseline:")
print(f"  RMSE: {rmse_improvement:.2f}% better")
print(f"  R²:   {r2_improvement:.2f}% better")

Training optimized XGBoost model...

Optimized XGBoost Performance (Validation Set):
  RMSE: 0.056288
  R²:   0.885255
  MAE:  0.043699

Improvement over Baseline:
  RMSE: 0.32% better
  R²:   0.08% better


## 6. CROSS-VALIDATION ON FULL TRAINING DATA

In [13]:
# Cross-validation scores on full training data
cv_scores = cross_val_score(
    optimized_model, X, y, cv=5, scoring='r2', n_jobs=-1
)

print(f"Cross-Validation Scores (5-fold):")
for i, score in enumerate(cv_scores):
    print(f"  Fold {i+1}: {score:.6f}")

print(f"\nMean CV R² Score: {cv_scores.mean():.6f} (+/- {cv_scores.std():.6f})")

Cross-Validation Scores (5-fold):
  Fold 1: 0.885064
  Fold 2: 0.887385
  Fold 3: 0.885753
  Fold 4: 0.885673
  Fold 5: 0.887158

Mean CV R² Score: 0.886207 (+/- 0.000904)


## 7. FEATURE IMPORTANCE ANALYSIS

In [14]:
# Get feature importances
feature_importance = pd.Series(
    optimized_model.feature_importances_, 
    index=X.columns
).sort_values(ascending=False)

print(f"\nTop 15 Most Important Features:")
print(feature_importance.head(15))

# Percentage importance
total_importance = feature_importance.sum()
cumulative_importance = feature_importance.cumsum() / total_importance

print(f"\nTop features cover:")
for n_features in [5, 10, 15]:
    coverage = cumulative_importance.iloc[n_features-1] * 100
    print(f"  Top {n_features} features: {coverage:.2f}% of model importance")


Top 15 Most Important Features:
lighting_night            0.578985
speed_limit               0.180048
curvature                 0.063878
weather_clear             0.053712
lighting_daylight         0.036181
weather_rainy             0.027141
num_reported_accidents    0.020410
lighting_dim              0.018539
weather_foggy             0.018032
holiday_True              0.000893
holiday_False             0.000806
public_road_False         0.000441
public_road_True          0.000435
road_type_urban           0.000208
road_type_highway         0.000162
dtype: float32

Top features cover:
  Top 5 features: 91.28% of model importance
  Top 10 features: 99.78% of model importance
  Top 15 features: 99.99% of model importance


## 8. FINAL PREDICTIONS ON TEST SET

In [17]:
optimized_model.fit(X, y)
test_predictions = optimized_model.predict(X_test)

## 9. GENERATE SUBMISSION FILE

In [18]:
from pathlib import Path

# Create submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'accident_risk': test_predictions
})

# Save to outputs folder
output_path = Path('./Outputs/submission_xgboost_optimized.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

submission.to_csv(output_path, index=False)

print(f"✓ Submission saved to {output_path}")
print(f"\nSubmission Preview (first 10 rows):")
print(submission.head(10))
print(f"\nSubmission shape: {submission.shape}")

✓ Submission saved to Outputs\submission_xgboost_optimized.csv

Submission Preview (first 10 rows):
       id  accident_risk
0  517754       0.288562
1  517755       0.124232
2  517756       0.186164
3  517757       0.322963
4  517758       0.403059
5  517759       0.460218
6  517760       0.262443
7  517761       0.199725
8  517762       0.360659
9  517763       0.318819

Submission shape: (172585, 2)


## 10. MODEL SUMMARY

In [19]:
print("="*60)
print("XGBOOST OPTIMIZED MODEL SUMMARY")
print("="*60)
print(f"\nOptimized Hyperparameters:")
for param, value in xgb_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nPerformance Metrics (Validation Set):")
print(f"  RMSE:        {opt_rmse:.6f}")
print(f"  R² Score:    {opt_r2:.6f}")
print(f"  MAE:         {opt_mae:.6f}")

print(f"\nCross-Validation (5-fold):")
print(f"  Mean R²:     {cv_scores.mean():.6f}")
print(f"  Std Dev:     {cv_scores.std():.6f}")

print(f"\nMost Important Feature: {feature_importance.index[0]}")
print(f"Feature Importance: {feature_importance.iloc[0]:.6f}")

print(f"\nDataset Information:")
print(f"  Training samples: {X_train.shape[0]}")
print(f"  Validation samples: {X_val.shape[0]}")
print(f"  Test samples: {X_test.shape[0]}")
print(f"  Features: {X.shape[1]}")

print("="*60)

XGBOOST OPTIMIZED MODEL SUMMARY

Optimized Hyperparameters:
  subsample: 0.6
  n_estimators: 100
  min_child_weight: 3
  max_depth: 7
  learning_rate: 0.1
  gamma: 0
  colsample_bytree: 0.6

Performance Metrics (Validation Set):
  RMSE:        0.056288
  R² Score:    0.885255
  MAE:         0.043699

Cross-Validation (5-fold):
  Mean R²:     0.886207
  Std Dev:     0.000904

Most Important Feature: lighting_night
Feature Importance: 0.578985

Dataset Information:
  Training samples: 414203
  Validation samples: 103551
  Test samples: 172585
  Features: 16
